# 🤖 Chatbot RAG para Consulta de Tarifas del Impuesto ICA

**Arquitectura independiente del extractor de PDFs:**
1. Cargar o genera la tabla estructurada de tarifas ICA (`Municipio`, `CIIU`, `Actividad`, `Tarifa`).
2. Se exporta la base normalizada en formato `.csv`.
3. Indexar los registros con embeddings locales (`all-MiniLM-L6-v2`) en **FAISS**.
4. Ejecuta un **workflow híbrido** (búsqueda exacta por CIIU/municipio + búsqueda semántica RAG).
5. Genera respuestas con **TinyLlama-1.1B-Chat** y expone un chat web con **Gradio**.
6. GPU T4.

## 🧾 Guia instalación de dependencias
## 📦 ejecutar en entorno GPU T4

In [ ]:
%pip install -q langchain langchain-community langchain-core langchain-huggingface
%pip install -q faiss-cpu sentence-transformers
%pip install -q transformers accelerate sentencepiece gradio pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 🧠 Carga de datos y exportación del CSV

In [ ]:
import pandas as pd
from google.colab import files

# 👇 Ojo Cambiar a True si para usar el archivo 'tarifas_ica.csv' generado por el notebook Proyecto tarifas del impuesto ICA.ipynb
SUBIR_CSV_PROPIO = False

if SUBIR_CSV_PROPIO:
    print("Seleccione el archivo CSV de tarifas ICA:")
    archivos = files.upload()
    nombre_csv = next(iter(archivos))
    df_tarifas = pd.read_csv(nombre_csv, dtype={"CIIU": str})
else:
    # Base de prueba basada en el Estatuto de San Vicente Ferrer y ejemplo de Bogotá
    datos_ejemplo = [
        {"Municipio": "San Vicente Ferrer", "CIIU": "6201", "Actividad": "Actividades de desarrollo de sistemas informáticos (planificación, análisis, diseño, programación)", "Tarifa": "8 x 1000"},
        {"Municipio": "San Vicente Ferrer", "CIIU": "2410", "Actividad": "Industrias básicas de hierro y de acero", "Tarifa": "7 x 1000"},
        {"Municipio": "San Vicente Ferrer", "CIIU": "4662", "Actividad": "Comercio al por mayor de metales y productos metalíferos", "Tarifa": "6 x 1000"},
        {"Municipio": "San Vicente Ferrer", "CIIU": "4711", "Actividad": "Comercio al por menor en establecimientos no especializados con surtido compuesto principalmente por alimentos", "Tarifa": "4 x 1000"},
        {"Municipio": "San Vicente Ferrer", "CIIU": "6920", "Actividad": "Actividades de contabilidad, teneduría de libros, auditoría financiera y asesoría tributaria", "Tarifa": "8 x 1000"},
        {"Municipio": "Bogotá", "CIIU": "6201", "Actividad": "Desarrollo de software", "Tarifa": "9.66 x 1000"},
    ]
    df_tarifas = pd.DataFrame(datos_ejemplo)

# Normalización de columnas clave
df_tarifas["CIIU"] = df_tarifas["CIIU"].astype(str).str.strip()
df_tarifas["Municipio"] = df_tarifas["Municipio"].astype(str).str.strip()

# Exportación del archivo CSV estructurado
archivo_salida = "tarifas_ica.csv"
df_tarifas.to_csv(archivo_salida, index=False, encoding="utf-8-sig")
print(f"Base consolidada con {len(df_tarifas)} registros y guardada en '{archivo_salida}'.")

# Descarga automática a tu equipo (descomenta la siguiente línea si deseas descargarlo al ejecutar)
# files.download(archivo_salida)

display(df_tarifas.head())

Base consolidada con 6 registros y guardada en 'tarifas_ica.csv'.


,Municipio,CIIU,Actividad,Tarifa
0,San Vicente Ferrer,6201,Actividades de desarrollo de sistemas informát...,8 x 1000
1,San Vicente Ferrer,2410,Industrias básicas de hierro y de acero,7 x 1000
2,San Vicente Ferrer,4662,Comercio al por mayor de metales y productos m...,6 x 1000
3,San Vicente Ferrer,4711,Comercio al por menor en establecimientos no e...,4 x 1000
4,San Vicente Ferrer,6920,"Actividades de contabilidad, teneduría de libr...",8 x 1000


## 🧠 Embeddings locales y base vectorial FAISS

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Construcción de documentos textuales y metadatos
texts = []
metadatas = []

for _, row in df_tarifas.iterrows():
    texto_doc = (
        f"Municipio: {row['Municipio']}. "
        f"Código CIIU: {row['CIIU']}. "
        f"Actividad económica: {row['Actividad']}. "
        f"Tarifa del impuesto ICA: {row['Tarifa']}."
    )
    texts.append(texto_doc)
    metadatas.append({
        "municipio": row["Municipio"],
        "ciiu": row["CIIU"],
        "tarifa": str(row["Tarifa"]),
    })

# Creación del índice FAISS en memoria
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

vectorstore = FAISS.from_texts(texts=texts, embedding=embeddings, metadatas=metadatas)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Índice FAISS creado correctamente con {len(texts)} documentos.")

/tmp/ipykernel_1061/1863469247.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Índice FAISS creado correctamente con 6 documentos.


## 🎯 Carga del LLM local (TinyLlama 1.1B)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo de ejecución:", device)

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.2,
    do_sample=True,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=gen_pipeline)
print("Modelo LLM cargado y listo.")

Dispositivo de ejecución: cuda


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo LLM cargado y listo.


## 🎯 Workflow Híbrido (CIIU + Municipio + RAG) y prueba

In [ ]:
import re
from langchain_core.prompts import PromptTemplate

template_str = """<|system|>
Eres un asistente tributario experto en el Impuesto de Industria y Comercio (ICA) en Colombia.
Usa EXCLUSIVAMENTE la información del CONTEXTO para responder de forma breve, clara y precisa en español.
Indica siempre el Municipio, el código CIIU y la Tarifa exacta.
Si el contexto no contiene la respuesta, indica: "No cuento con información para esa consulta en la base cargada."
<|user|>
CONTEXTO:
{context}

PREGUNTA:
{question}
<|assistant|>
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template_str,
)

def detectar_municipio(pregunta: str, df: pd.DataFrame):
    pregunta_lower = pregunta.lower()
    for mun in df["Municipio"].unique():
        if mun.lower() in pregunta_lower:
            return mun
    return None

def responder_consulta_ica(pregunta: str) -> str:
    codigos_ciiu = re.findall(r"\b\d{4}\b", pregunta)
    municipio_detectado = detectar_municipio(pregunta, df_tarifas)
    contexto = ""

    # Ruta 1: Filtro determinista exacto por CIIU (y municipio si se menciona)
    if codigos_ciiu:
        filtro = df_tarifas[df_tarifas["CIIU"].isin(codigos_ciiu)]
        if municipio_detectado:
            filtro_mun = filtro[filtro["Municipio"] == municipio_detectado]
            if not filtro_mun.empty:
                filtro = filtro_mun

        if not filtro.empty:
            lineas = [
                f"Municipio: {r['Municipio']}. Código CIIU: {r['CIIU']}. Actividad: {r['Actividad']}. Tarifa ICA: {r['Tarifa']}."
                for _, r in filtro.iterrows()
            ]
            contexto = "\n".join(lineas)

    # Ruta 2: Recuperación semántica RAG con FAISS
    if not contexto:
        docs = retriever.invoke(pregunta)
        if municipio_detectado:
            docs_mun = [d for d in docs if d.metadata.get("municipio") == municipio_detectado]
            docs = docs_mun if docs_mun else docs
        contexto = "\n".join(d.page_content for d in docs)

    # Generación de respuesta con el LLM
    prompt_final = prompt.format(context=contexto, question=pregunta)
    respuesta = llm.invoke(prompt_final)
    return respuesta.strip()

# Prueba rápida en consola
print("Prueba 1:", responder_consulta_ica("¿Cuál es la tarifa ICA para el código CIIU 6201 en Bogotá?"))
print("\nPrueba 2:", responder_consulta_ica("¿Qué tarifa aplica para industrias básicas de hierro y acero?"))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prueba 1: Sí, la tarifa ICA para el código CIIU 6201 en Bogotá es de 9.66 x 1000.

Prueba 2: La tarifa del impuesto ICA para industrias básicas de hierro y acero es de 7 x 1000.


## 📔🌐   Interfaz web del Chatbot con Gradio

In [ ]:
import gradio as gr

def gradio_ica_fn(message, history):
    return responder_consulta_ica(message)

chat_interface = gr.ChatInterface(
    fn=gradio_ica_fn,
    title="Chatbot de Consulta de Tarifas ICA (RAG + FAISS)",
    description="Consulta tarifas del Impuesto de Industria y Comercio por código CIIU, actividad económica o municipio.",
    examples=[
        "¿Cuál es la tarifa ICA para el CIIU 6201 en San Vicente Ferrer?",
        "¿Cuál es la tarifa ICA para el CIIU 6201 en Bogotá?",
        "¿Qué tarifa aplica para industrias básicas de hierro y acero?",
        "¿Cuál es la tarifa para actividades de contabilidad y asesoría tributaria?"
    ]
)

chat_interface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://74cd068ec723ea14cb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://74cd068ec723ea14cb.gradio.live
